# RQ3: Runtime Requirements

**Research Question**: What are the runtime requirements of the generalization approach?

This notebook analyzes the runtime requirements for Teralizer tool execution across different configurations:
- **Total Runtime**: Teralizer execution time per project
- **Pipeline Stages**: Runtime breakdown by processing stage and variant
- **Efficiency Comparison**: Pareto front analysis comparing EvoSuite and Teralizer
- **EvoSuite Analysis**: Runtime breakdown by phase and search budget

In [ ]:
from teralizer.config import db_config
from teralizer.rq3_runtime_requirements import (
    get_teralizer_total_runtimes,
    get_teralizer_runtime_by_stage,
    get_evosuite_vs_teralizer_efficiency,
    get_evosuite_runtime_analysis,
    compute_teralizer_runtime_statistics,
    compute_stage_runtime_breakdown,
    compute_pareto_efficiency_analysis,
    compute_evosuite_phase_statistics,
    generate_teralizer_runtimes_table,
    generate_runtime_breakdown_csv,
    generate_pareto_efficiency_csv,
    generate_evosuite_runtime_csv,
    generate_teralizer_runtimes_csv,
    generate_pareto_points_table,
)
from teralizer.exports import (
    save_latex_table,
    save_csv_data,
    save_figure,
    get_table_group_order,
    get_project_within_type_order,
)
from teralizer.plotting import setup_paper_style
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# Database connection
conn = db_config.get_dev_engine()

# Configure paper style
setup_paper_style()

## Total Teralizer Runtime per Project

Establishes the overall runtime baseline by showing total execution time required for each project.

In [ ]:
# Get and process total runtime data
df_total_runtimes = get_teralizer_total_runtimes(conn)
df_total_runtimes_processed = compute_teralizer_runtime_statistics(df_total_runtimes)

# Generate and save LaTeX table
latex_table = generate_teralizer_runtimes_table(df_total_runtimes_processed)
save_latex_table(latex_table, "tab-teralizer-runtimes")

# Display the table
print(latex_table)

# Display data for reference
display_df = df_total_runtimes_processed[["project_name", "runtime"]].copy()
display_df["runtime_formatted"] = display_df["runtime"].apply(
    lambda x: f"{int(x // 3600)}h {int((x % 3600) // 60):02d}min {int(x % 60):02d}s"
)
display_df = display_df.rename(
    columns={"project_name": "Project", "runtime_formatted": "Runtime"}
)
display(display_df[["Project", "Runtime"]])

# Export CSV data
csv_data = generate_teralizer_runtimes_csv(df_total_runtimes_processed)
save_csv_data(csv_data, "teralizer-total-runtimes")

## Runtime Breakdown by Pipeline Stage

Detailed analysis of runtime distribution across different pipeline stages and generalization variants.

In [ ]:
# Get and process stage runtime data
df_stage_runtimes = get_teralizer_runtime_by_stage(conn)
df_stage_processed = compute_stage_runtime_breakdown(df_stage_runtimes)

# Get unique variants across all data, ordered by variant_order
variant_order_map = (
    df_stage_processed.drop_duplicates("variant")
    .set_index("variant")["variant_order"]
    .to_dict()
)
all_variants = sorted(
    df_stage_processed["variant"].unique(),
    key=lambda v: variant_order_map.get(v, float("inf")),
)
non_shared_variants = [v for v in all_variants if v != "SHARED"]

# Calculate total runtime per base project for sorting
base_project_totals = (
    df_stage_processed.groupby("base_project_name")["total_runtime"]
    .sum()
    .sort_values(ascending=False)
)
top_base_projects = base_project_totals.head(10).index.tolist()

# Filter for top base projects
top_projects_data = df_stage_processed[
    df_stage_processed["base_project_name"].isin(top_base_projects)
].copy()

# Create distinct color mapping
distinct_colors = [
    "#1f77b4",
    "#ff7f0e",
    "#2ca02c",
    "#d62728",
    "#9467bd",
    "#8c564b",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
    "#17becf",
    "#aec7e8",
    "#ffbb78",
    "#98df8a",
    "#ff9896",
    "#c5b0d5",
]

if len(all_variants) > len(distinct_colors):
    additional_colors = plt.cm.get_cmap("Set3")(
        np.linspace(0, 1, len(all_variants) - len(distinct_colors))
    )
    additional_colors = [tuple(c) for c in additional_colors]
    distinct_colors.extend(additional_colors)

color_map = {variant: distinct_colors[i] for i, variant in enumerate(all_variants)}

# Define which variants apply to which stage groups, ordered by variant_order
ordered_groups = [
    "Original Validation",
    "Specification Extraction",
    "Initial Validation",
    "Test Transformation",
    "Generalization Validation",
]

stage_group_variants = {
    "Original Validation": ["SHARED"],
    "Specification Extraction": ["SHARED"],
    "Initial Validation": ["SHARED"],
    "Test Transformation": sorted(
        non_shared_variants, key=lambda v: variant_order_map.get(v, float("inf"))
    ),
    "Generalization Validation": sorted(
        non_shared_variants, key=lambda v: variant_order_map.get(v, float("inf"))
    ),
}

# Get unique project IDs for plotting, sorted by table order
project_within_type_order = get_project_within_type_order()
project_id_to_name = (
    top_projects_data.drop_duplicates("project_id")
    .set_index("project_id")["project_name"]
    .to_dict()
)
unique_project_ids = list(top_projects_data["project_id"].unique())

# Sort by table order
unique_project_ids = sorted(
    unique_project_ids,
    key=lambda pid: (
        get_table_group_order(project_id_to_name[pid], "INITIAL"),
        project_within_type_order.get(project_id_to_name[pid], 99),
    ),
)

print(
    f"Processing {len(unique_project_ids)} projects with {len(all_variants)} variants"
)
print(f"Stage groups: {ordered_groups}")
print(f"All variants: {all_variants}")

In [ ]:
# Define parameters for bar positioning
bar_width = 0.3
bar_spacing = 0.05
group_spacing = 0.3

# Count the number of bars in each group
bars_per_group = {
    group: len(variants) for group, variants in stage_group_variants.items()
}

# Calculate the total width of each group
group_widths = {
    group: (count * bar_width) + ((count - 1) * bar_spacing) if count > 0 else 0
    for group, count in bars_per_group.items()
}

# Calculate the center position of each group
group_centers = {}
current_position = 0
for group in ordered_groups:
    width = group_widths[group]
    group_centers[group] = current_position + width / 2
    current_position += width + group_spacing

# Calculate the position of each bar within its group
bar_positions = {}
for group in ordered_groups:
    variants = stage_group_variants[group]
    num_bars = len(variants)
    if num_bars == 0:
        continue
    group_center = group_centers[group]
    group_width = group_widths[group]
    start_pos = group_center - group_width / 2
    for i, variant in enumerate(variants):
        bar_positions[(group, variant)] = (
            start_pos + i * (bar_width + bar_spacing) + bar_width / 2
        )

# Create pivot tables for all projects
all_pivot_tables = {}
for project_id in unique_project_ids:
    project_data = top_projects_data[top_projects_data["project_id"] == project_id]
    pivot_data = pd.pivot_table(
        project_data,
        index="stage_group",
        columns="variant",
        values="total_runtime",
        aggfunc="sum",
        fill_value=0,
        observed=False,
    )
    all_pivot_tables[project_id] = pivot_data

# Find the maximum bar height for each base project
base_project_max_values = {}
for base_name in top_projects_data["base_project_name"].unique():
    max_value = 0
    for project_id in unique_project_ids:
        project_data = top_projects_data[top_projects_data["project_id"] == project_id]
        if project_data.empty:
            continue
        if project_data["base_project_name"].iloc[0] == base_name:
            pivot_data = all_pivot_tables[project_id]
            if not pivot_data.empty:
                project_max = pivot_data.max().max()
                max_value = max(max_value, project_max)
    base_project_max_values[base_name] = max_value * 1.2


# Helper function to format runtime values
def format_runtime(seconds):
    if seconds < 10:
        return f"{seconds:.1f}"
    elif seconds < 100:
        return f"{seconds:.0f}"
    elif seconds < 1000:
        return f"{seconds:.0f}"
    else:
        return f"{seconds / 1000:.1f}k"


# Prepare multi-line x-tick labels
xtick_labels = [
    "Original\nValidation"
    if g == "Original Validation"
    else "Specification\nExtraction"
    if g == "Specification Extraction"
    else "Initial\nValidation"
    if g == "Initial Validation"
    else "Test Transformation\n(BASELINE, NAIVE$_{10|50|200}$, IMPROVED$_{10|50|200}$)"
    if g == "Test Transformation"
    else "Generalization Validation\n(BASELINE, NAIVE$_{10|50|200}$, IMPROVED$_{10|50|200}$)"
    if g == "Generalization Validation"
    else g
    for g in ordered_groups
]

print(
    f"Bar positioning setup complete. Total x-axis width: {current_position - group_spacing + 0.4}"
)

In [ ]:
# Plot runtime by stage group for each project
fig1 = plt.figure(figsize=(17, 3.1 * len(unique_project_ids)))
plt.subplots_adjust(hspace=0.3)

for i, project_id in enumerate(unique_project_ids):
    project_data = top_projects_data[top_projects_data["project_id"] == project_id]
    if project_data.empty:
        continue
    project_name = project_data["project_name"].iloc[0]
    base_name = project_data["base_project_name"].iloc[0]
    ax = plt.subplot(len(unique_project_ids), 1, i + 1)
    pivot_data = all_pivot_tables[project_id]

    for group in ordered_groups:
        variants = stage_group_variants[group]
        for variant in variants:
            if variant in pivot_data.columns and group in pivot_data.index:
                value = pivot_data.loc[group, variant]
                position = bar_positions[(group, variant)]
                if value > 0:
                    bar = ax.bar(
                        position, value, width=bar_width, color=color_map[variant]
                    )
                    formatted_value = format_runtime(value)
                    ax.text(
                        position,
                        value + (base_project_max_values[base_name] * 0.02),
                        formatted_value,
                        ha="center",
                        va="bottom",
                        fontsize=16,
                        rotation=0,
                    )

    ax.set_title(f"Project: {project_name}")
    ax.set_ylabel("Runtime (s)")
    ax.set_xticks([group_centers[group] for group in ordered_groups])
    if i == len(unique_project_ids) - 1:
        ax.set_xticklabels(xtick_labels, rotation=0, ha="center")
    else:
        ax.set_xticklabels([])
    ax.set_xlim(-0.2, current_position - group_spacing + 0.2)
    ax.set_ylim(0, base_project_max_values[base_name])
    if ax.get_legend() is not None:
        ax.get_legend().remove()


# Add horizontal legend at the top
def prettify_variant_label(label):
    return re.sub(r"_(\d+)_TRIES$", r"$_{\1}$", label)


legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=color_map[variant]) for variant in all_variants
]
legend_labels = [prettify_variant_label(label) for label in all_variants]

fig1.legend(
    legend_handles,
    legend_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.0),
    frameon=False,
    ncol=len(legend_labels),
)

plt.tight_layout()
plt.subplots_adjust(top=0.95, right=1)
save_figure(fig1, "fig_teralizer_runtimes")
plt.show()

# Export CSV data
csv_data = generate_runtime_breakdown_csv(df_stage_processed)
save_csv_data(csv_data, "runtime-breakdown-data")

## EvoSuite vs Teralizer Efficiency Comparison

Pareto front analysis comparing the efficiency trade-offs between EvoSuite-only and EvoSuite+Teralizer approaches.

In [ ]:
# Get and process efficiency comparison data
df_efficiency = get_evosuite_vs_teralizer_efficiency(conn)

if not df_efficiency.empty:
    df_pareto = compute_pareto_efficiency_analysis(df_efficiency)

    # Show summary of the data
    print(f"Efficiency comparison data shape: {df_efficiency.shape}")
    print(f"Unique projects: {df_efficiency['project_name'].nunique()}")
    print(f"Unique variants: {df_efficiency['teralizer_variant'].unique()}")

    display(df_efficiency.head())

    print(f"\nPareto efficiency analysis shape: {df_pareto.shape}")
    print(f"Pareto optimal points: {df_pareto['is_pareto_optimal'].sum()}")

    # Export CSV data
    csv_data = generate_pareto_efficiency_csv(df_pareto)
    save_csv_data(csv_data, "pareto-efficiency-data")

else:
    print("No efficiency comparison data available")

In [ ]:
# Create Pareto efficiency visualization and tables
if not df_efficiency.empty:

    def extract_prefix_and_budget(name):
        match = re.match(r"(.+)-es-default-(\d+s)", name)
        if match:
            return match.group(1), match.group(2)
        return name, None

    def pareto_front(df_points, x_col, y_col):
        sorted_df = df_points.sort_values(x_col)
        pareto = []
        max_y = -float("inf")
        for _, row in sorted_df.iterrows():
            if row[y_col] > max_y:
                pareto.append(row)
                max_y = row[y_col]
        return pd.DataFrame(pareto)

    # Create a DataFrame with project_prefix and search_budget
    prefix_budget = df_efficiency["project_name"].apply(
        lambda x: pd.Series(extract_prefix_and_budget(x))
    )
    prefix_budget.columns = ["project_prefix", "search_budget"]
    df_temp = pd.concat([df_efficiency.reset_index(drop=True), prefix_budget], axis=1)

    project_prefixes = df_temp["project_prefix"].unique()

    # Ensure EqBench comes first for consistent ordering
    if "eqbench" in project_prefixes and "commons-utils" in project_prefixes:
        project_prefixes = ["eqbench", "commons-utils"]
    else:
        project_prefixes = sorted(project_prefixes)

    # Prepare evosuite_points
    evosuite_points = (
        df_temp.groupby(["project_prefix", "search_budget"]).first().reset_index()
    )

    # Create combined figure
    fig, axes = plt.subplots(nrows=1, ncols=len(project_prefixes), figsize=(9, 3.5))
    if len(project_prefixes) == 1:
        axes = [axes]

    for ax, project in zip(axes, project_prefixes):
        proj_df = df_temp[df_temp["project_prefix"] == project].copy()
        evosuite_df = evosuite_points[
            evosuite_points["project_prefix"] == project
        ].copy()

        # Plot all points (faded)
        ax.scatter(
            evosuite_df["evosuite_runtime"],
            evosuite_df["evosuite_detected"],
            marker="o",
            color="blue",
            alpha=0.3,
            s=40,
            label="EvoSuite only",
        )
        naive_mask = proj_df["teralizer_variant"].str.startswith("NAIVE")
        ax.scatter(
            proj_df[naive_mask]["total_runtime"],
            proj_df[naive_mask]["teralizer_detected"],
            marker="x",
            color="red",
            alpha=0.3,
            s=40,
            label="EvoSuite + NAIVE",
        )
        improved_mask = proj_df["teralizer_variant"].str.startswith("IMPROVED")
        ax.scatter(
            proj_df[improved_mask]["total_runtime"],
            proj_df[improved_mask]["teralizer_detected"],
            marker="^",
            color="green",
            alpha=0.3,
            s=40,
            label="EvoSuite + IMPROVED",
        )

        # Prepare all points for Pareto front
        es_points = evosuite_df[
            ["evosuite_runtime", "evosuite_detected", "search_budget"]
        ].copy()
        es_points["type"] = "ES"
        es_points["variant"] = None
        es_points.rename(
            columns={"evosuite_runtime": "runtime", "evosuite_detected": "detected"},
            inplace=True,
        )

        naive_points = proj_df[naive_mask][
            [
                "total_runtime",
                "teralizer_detected",
                "search_budget",
                "teralizer_variant",
            ]
        ].copy()
        naive_points["type"] = "NAIVE"
        naive_points.rename(
            columns={"total_runtime": "runtime", "teralizer_detected": "detected"},
            inplace=True,
        )

        improved_points = proj_df[improved_mask][
            [
                "total_runtime",
                "teralizer_detected",
                "search_budget",
                "teralizer_variant",
            ]
        ].copy()
        improved_points["type"] = "IMPROVED"
        improved_points.rename(
            columns={"total_runtime": "runtime", "teralizer_detected": "detected"},
            inplace=True,
        )

        all_points = pd.concat(
            [es_points, naive_points, improved_points], ignore_index=True, sort=False
        )
        pf = pareto_front(all_points, "runtime", "detected")

        # Axis formatting
        y_data_min = min(all_points["detected"].min(), pf["detected"].min())
        y_data_max = max(all_points["detected"].max(), pf["detected"].max())
        y_range = y_data_max - y_data_min
        margin = 0.2 * y_range if y_range > 0 else 0.5
        y_min = y_data_min - margin / 2
        y_max = y_data_max + margin
        ax.set_ylim(y_min, y_max)

        ax.set_title(f"Project: {project}", fontsize=14)
        ax.set_xlabel("Runtime (s)", fontsize=12)
        ax.set_ylabel("Detected (%)", fontsize=12)
        ax.tick_params(axis="both", which="major", labelsize=10)
        ax.ticklabel_format(style="plain", axis="x")

        # Draw Pareto front line
        ax.plot(
            pf["runtime"],
            pf["detected"],
            linestyle="--",
            color="black",
            linewidth=1.2,
            zorder=2,
            label="Pareto front",
        )

        # Plot and label Pareto front points
        offset = 0.025 * (y_max - y_min)
        for i, (_, row) in enumerate(pf.iterrows(), start=1):
            # Plotting
            if row["type"] == "ES":
                color = "blue"
                marker = "o"
            elif row["type"] == "NAIVE":
                color = "red"
                marker = "x"
            elif row["type"] == "IMPROVED":
                color = "green"
                marker = "^"
            else:
                color = "black"
                marker = "o"

            if marker in ["o", "^"]:
                ax.scatter(
                    row["runtime"],
                    row["detected"],
                    marker=marker,
                    color=color,
                    s=90,
                    edgecolor="black",
                    zorder=3,
                )
            else:
                ax.scatter(
                    row["runtime"],
                    row["detected"],
                    marker=marker,
                    color=color,
                    s=90,
                    zorder=3,
                )

            ax.text(
                row["runtime"],
                row["detected"] + offset,
                str(i),
                fontsize=14,
                fontweight="bold",
                color=color,
                ha="center",
                va="bottom",
            )

    # Only show one legend, outside the plot area
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, fontsize=10)

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    save_figure(fig, "fig_teralizer_efficiency")
    plt.show()

    print(
        f"Generated combined Pareto efficiency plot for {len(project_prefixes)} projects"
    )

    # Generate separate tables for each project
    unique_projects = df_pareto["project_name"].unique()
    print(f"Generating separate Pareto tables for projects: {unique_projects}")

    for project_name in unique_projects:
        try:
            table_content = generate_pareto_points_table(df_pareto, project_name)

            # Determine output filename based on project name
            if "eqbench" in project_name.lower():
                table_filename = "tab-pareto-eqbench"
            elif "commons" in project_name.lower():
                table_filename = "tab-pareto-commons"
            else:
                # Fallback for unexpected project names
                clean_name = project_name.lower().replace("-", "").replace("_", "")
                table_filename = f"tab-pareto-{clean_name}"

            save_latex_table(table_content, table_filename)
            print(f"Saved table: {table_filename}.tex")

        except Exception as e:
            print(f"Error generating table for {project_name}: {e}")

else:
    print("Skipping Pareto efficiency visualization - no data available")

## EvoSuite Runtime Analysis by Phase

Analysis of EvoSuite runtime breakdown across different execution phases and search budget configurations.

In [ ]:
# Get and process EvoSuite runtime data
df_evosuite = get_evosuite_runtime_analysis(conn)
df_evosuite_processed = compute_evosuite_phase_statistics(df_evosuite)

print(f"EvoSuite runtime data shape: {df_evosuite_processed.shape}")
print(f"Unique projects: {df_evosuite_processed['project_name'].nunique()}")
print(f"Search budgets: {sorted(df_evosuite_processed['search_budget'].unique())}")

# Define the runtime columns to analyze
runtime_columns = [
    "total",
    "search",
    "inlining",
    "minimization",
    "coverage_analysis",
    "assertion_generation",
    "junit_check",
    "writing_tests",
    "writing_statistics",
    "done",
    "finished",
]

# Create mean and median dataframes by search budget
mean_df = (
    df_evosuite_processed.groupby("search_budget")[runtime_columns].mean().reset_index()
)
mean_df = mean_df.sort_values("search_budget")

median_df = (
    df_evosuite_processed.groupby("search_budget")[runtime_columns]
    .median()
    .reset_index()
)
median_df = median_df.sort_values("search_budget")

print("\nMean EvoSuite runtimes per class:")
display(mean_df)

print("\nMedian EvoSuite runtimes per class:")
display(median_df)

# Export CSV data
csv_data = generate_evosuite_runtime_csv(df_evosuite_processed)
save_csv_data(csv_data, "evosuite-runtime-analysis")

In [ ]:
# Create EvoSuite phase analysis visualization
if not mean_df.empty:
    # Set search_budget as index
    mean_df_indexed = mean_df.set_index("search_budget")

    # Select only the phase columns (exclude 'total' as it's the sum of all phases)
    phase_columns = [
        "search",
        "inlining",
        "minimization",
        "coverage_analysis",
        "assertion_generation",
        "junit_check",
        "writing_tests",
        "writing_statistics",
        "done",
        "finished",
    ]

    # Transpose to get phases as rows and search budgets as columns
    plot_data = mean_df_indexed[phase_columns].transpose()

    # Create the plot
    fig, ax = plt.subplots(figsize=(16, 4))

    # Get the number of phases and search budgets
    n_phases = len(phase_columns)
    n_budgets = len(plot_data.columns)

    # Set the width of each bar and the spacing between groups
    bar_width = 0.8 / n_budgets
    group_spacing = np.arange(n_phases)

    # Use tab10 colormap
    colors = plt.cm.get_cmap("tab10")(np.arange(n_budgets) % 10)

    # Find the maximum value to adjust y-axis limits
    max_value = plot_data.max().max()

    # Plot each search budget as a set of bars
    for i, (budget, color) in enumerate(zip(plot_data.columns, colors)):
        positions = group_spacing + (i - n_budgets / 2 + 0.5) * bar_width
        bars = ax.bar(
            positions,
            plot_data[budget],
            bar_width,
            label=f"Budget: {budget}s",
            color=color,
        )

        # Add text labels on top of each bar
        for bar_idx, bar in enumerate(bars):
            height = bar.get_height()
            # Format value to 1 decimal place if it's >= 1, otherwise 2 decimal places
            if height >= 1:
                value_text = f"{height:.1f}"
            else:
                value_text = f"{height:.2f}"

            # Position the text above the bar
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 1,
                value_text,
                ha="center",
                va="bottom",
                fontsize=8,
            )

    # Set the x-axis labels and positions
    ax.set_xticks(group_spacing)
    ax.set_xticklabels(phase_columns, rotation=45, ha="right")

    # Add labels and title
    ax.set_ylabel("Mean Runtime (seconds)")
    ax.set_title("Mean EvoSuite Runtime by Phase and Search Budget")

    # Add a legend
    ax.legend(title="Search Budget (seconds)")

    # Add grid lines for better readability
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    # Set y-axis limit to add more space for labels (add 15% padding)
    ax.set_ylim(0, max_value * 1.15)

    # Adjust layout to make room for labels
    plt.tight_layout()
    save_figure(fig, "fig_evosuite_runtime_phases")
    plt.show()

    print(
        f"Generated EvoSuite phase analysis for {n_budgets} search budgets and {n_phases} phases"
    )

else:
    print("No EvoSuite runtime data available for visualization")

## RQ3 Analysis Complete

Generated outputs:
- `tab-teralizer-runtimes.tex` - Total runtime per project
- `tab-pareto-eqbench.tex` - Pareto optimal points for EqBench project
- `tab-pareto-commons.tex` - Pareto optimal points for Commons-utils project
- `fig_teralizer_runtimes.pdf` - Runtime breakdown by pipeline stage and variant
- `fig_teralizer_efficiency.pdf` - Pareto efficiency comparison visualization
- `fig_evosuite_runtime_phases.pdf` - EvoSuite phase analysis by search budget
- Corresponding CSV data files for all analyses